In [ ]:
labs

In [ ]:
import pandas as pd
import numpy as np
import gcsfs
import gc

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
clean_dir = f"{bucket}/amia/clean_data"
feature_dir = f"{bucket}/amia/feature"
FREQ_THRESHOLD = 0.005


def build_unit_maps():
    main_unit_map = {
        '10839-9': 'nanogram per milliliter', '10976-9': 'no_unit',
        '13457-7': 'milligram per deciliter', '13458-5': 'milligram per deciliter',
        '14314-9': 'no_unit', '14316-4': 'no_unit',
        '1742-6':  'unit per liter', '1751-7':  'gram per liter',
        '18282-4': 'no_unit', '1920-8':  'unit per liter',
        '19295-5': 'no_unit', '19359-9': 'no_unit',
        '1960-4':  'millimole per liter', '19642-8': 'no_unit',
        '19659-2': 'no_unit', '1968-7':  'milligram per deciliter',
        '1975-2':  'milligram per deciliter', '1988-5':  'milligram per liter',
        '2019-8':  'millimeter mercury column', '20507-0': 'no_unit',
        '2085-9':  'milligram per deciliter', '2093-3':  'milligram per deciliter',
        '2132-9':  'picogram per milliliter', '2157-6':  'unit per liter',
        '2160-0':  'milligram per deciliter', '2284-8':  'nanogram per milliliter',
        '2524-7':  'millimole per liter', '2571-8':  'milligram per deciliter',
        '2703-7':  'millimeter mercury column', '2744-1':  'no_unit',
        '2885-2':  'gram per deciliter', '2965-2':  'no_unit',
        '30341-2': 'millimeter per hour', '3094-0':  'milligram per deciliter',
        '32693-4': 'millimole per liter', '3349-8':  'no_unit',
        '3414-0':  'no_unit', '3773-9':  'no_unit',
        '40487-1': 'no_unit', '41653-7': 'milligram per deciliter',
        '43396-1': 'milligram per deciliter', '43402-7': 'millimeter per hour',
        '4537-7':  'millimeter per hour', '4548-4':  'percent',
        '5643-2':  'milligram per deciliter', '5802-4':  'no_unit',
        '5811-5':  'no_unit', '5902-2':  'second',
        '59408-5': 'percent', '59673-4': 'no_unit',
        '6301-6':  'no_unit', '6598-7':  'nanogram per milliliter',
        '6768-6':  'unit per liter', '6873-4':  'millimole per liter',
        '76492-8': 'no_unit', '777-3':   'thousand per microliter',
        '8310-5':  'degree Celsius', '8331-1':  'degree Fahrenheit',
        '9279-1':  'per minute', '9564003': 'no_unit',
    }

    rename_map = {
        '1742-6':  {'IU/L': 'unit per liter',
                    'international unit per milliliter': 'unit per liter'},
        '1920-8':  {'IU/L': 'unit per liter',
                    'international unit per milliliter': 'unit per liter'},
        '6768-6':  {'IU/L': 'unit per liter',
                    'international unit per milliliter': 'unit per liter'},
        '2157-6':  {'U/L': 'unit per liter', 'IU/L': 'unit per liter',
                    'international unit per liter': 'unit per liter'},
        '4548-4':  {'Percent': 'percent', 'Percentage unit': 'percent',
                    'Of Total H': 'percent',
                    'percent total protein': 'percent',
                    'percent hemoglobin': 'percent',
                    'Percentage total hemoglobin': 'percent'},
        '2019-8':  {'mmHg': 'millimeter mercury column',
                    'torr': 'millimeter mercury column'},
        '2703-7':  {'mmHg': 'millimeter mercury column',
                    'torr': 'millimeter mercury column'},
        '1960-4':  {'milliequivalent per liter': 'millimole per liter',
                    'mEq/L': 'millimole per liter',
                    'mmol/L': 'millimole per liter'},
        '2524-7':  {'milliequivalent per liter': 'millimole per liter',
                    'mEq/L': 'millimole per liter',
                    'mmol/L': 'millimole per liter'},
        '32693-4': {'milliequivalent per liter': 'millimole per liter'},
        '2132-9':  {'pg/mL': 'picogram per milliliter'},
        '1968-7':  {'mg/dL': 'milligram per deciliter'},
        '1988-5':  {'mg/L': 'milligram per liter'},
        '4537-7':  {'mm/h': 'millimeter per hour'},
        '5902-2':  {'Seconds': 'second'},
        '8310-5':  {'degrees C': 'degree Celsius'},
        '13457-7': {'milligram per deciliter calculated': 'milligram per deciliter'},
        '43396-1': {'milligram per deciliter calculated': 'milligram per deciliter',
                    'mg/dL': 'milligram per deciliter'},
        '2885-2':  {'gram per deciliter calculated': 'gram per deciliter'},
        '9279-1':  {'minute': 'per minute',
                    'Respiratory rate': 'per minute'},
    }

    convert_map = {
        '13457-7': {'milligram per milliliter': ('milligram per deciliter', 100)},
        '13458-5': {'milligram per milliliter': ('milligram per deciliter', 100)},
        '2085-9':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '2093-3':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '2571-8':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '1968-7':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '1975-2':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '2160-0':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '3094-0':  {'milligram per milliliter': ('milligram per deciliter', 100)},
        '43396-1': {'milligram per milliliter': ('milligram per deciliter', 100)},
        '1988-5':  {'milligram per deciliter': ('milligram per liter', 10)},
        '10839-9': {'picogram per milliliter': ('nanogram per milliliter', 0.001),
                    'nanogram per liter':       ('nanogram per milliliter', 0.001),
                    'microgram per liter':      ('nanogram per milliliter', 1.0)},
        '6598-7':  {'picogram per milliliter': ('nanogram per milliliter', 0.001),
                    'nanogram per liter':       ('nanogram per milliliter', 0.001)},
        '1751-7':  {'milligram per liter':     ('gram per liter', 0.001),
                    'milligram per milliliter': ('gram per liter', 1.0)},
        '41653-7': {'gram per deciliter': ('milligram per deciliter', 1000)},
        '8331-1':  {'degree Celsius': ('degree Fahrenheit', 'C_to_F')},
        '6873-4':  {'milligram per deciliter': ('millimole per liter', 0.0961)},
    }

    return main_unit_map, rename_map, convert_map


POSITIVE_TERMS = {'Positive', 'Detected', 'Presumptive positive', 'Present', 'Reactive'}
NEGATIVE_TERMS = {'Negative', 'Not detected', 'Not detected/negative', 'None detected',
                  'Nonreactive', 'Non-Reactive', 'Not reactive to light'}

def label_concept_value(v):
    if pd.isna(v):
        return 'unknown'
    if v in POSITIVE_TERMS:
        return 'pos'
    if v in NEGATIVE_TERMS:
        return 'neg'
    return 'unknown'


PURE_CONCEPT_CODES = {'19295-5', '9564003', '18282-4', '19642-8', '19359-9',
                      '20507-0', '3773-9', '3414-0', '76492-8', '59673-4',
                      '10976-9'}
CBC_CODE = '9564003'


def load_anchors(timeframe):
    pos = pd.read_csv(f"{clean_dir}/clean_positive_{timeframe}.csv",
                      usecols=['person_id', 'index_date', 'timeframe_start'])
    neg = pd.read_csv(f"{clean_dir}/clean_negative_anchor_{timeframe}.csv",
                      usecols=['person_id', 'index_date', 'timeframe_start'])
    pos['IsPositive'] = 1
    neg['IsPositive'] = 0
    anchors = pd.concat([pos, neg], axis=0, ignore_index=True)
    anchors['index_date'] = pd.to_datetime(anchors['index_date'])
    anchors['timeframe_start'] = pd.to_datetime(anchors['timeframe_start'])
    print(f"[TF={timeframe}] Anchors: {len(anchors):,} 人")
    return anchors


def load_lab_table(timeframe, chunk_size=200_000):
    cols = ['person_id', 'standard_concept_name', 'standard_concept_code',
            'standard_vocabulary', 'measurement_datetime',
            'value_as_number', 'value_as_concept_name', 'unit_concept_name']
    dtype_spec = {
        'person_id': 'int64',
        'standard_concept_name': 'string',
        'standard_concept_code': 'string',
        'standard_vocabulary': 'string',
        'value_as_number': 'float32',
        'value_as_concept_name': 'string',
        'unit_concept_name': 'string',
    }
    cat_cols = ['standard_concept_code', 'standard_vocabulary',
                'unit_concept_name', 'value_as_concept_name',
                'standard_concept_name']

    chunks = []
    n_total_raw = 0
    n_total_kept = 0

    for path in [f"{clean_dir}/clean_lab_{timeframe}.csv",
                 f"{clean_dir}/clean_negative_lab_{timeframe}.csv"]:
        print(f"Reading {path} ...")
        n_file_raw = 0
        n_file_kept = 0

        for chunk in pd.read_csv(path, usecols=cols, chunksize=chunk_size,
                                 dtype=dtype_spec, low_memory=False):
            n_file_raw += len(chunk)
            chunk['measurement_datetime'] = pd.to_datetime(
                chunk['measurement_datetime'], errors='coerce')
            chunk = chunk[chunk['measurement_datetime'].notna()]
            for c in cat_cols:
                chunk[c] = chunk[c].astype('category')
            n_file_kept += len(chunk)
            chunks.append(chunk)

        print(f"  original: {n_file_raw:,} → keep: {n_file_kept:,} "
              f"(drop {n_file_raw - n_file_kept:,} NA time)")
        n_total_raw += n_file_raw
        n_total_kept += n_file_kept

    df = pd.concat(chunks, axis=0, ignore_index=True)
    del chunks
    gc.collect()

    for c in cat_cols:
        df[c] = df[c].astype('category')

    print(f"[TF={timeframe}] mergeing done: total original {n_total_raw:,}, keep {n_total_kept:,} 行")
    print(f"  storage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    return df


def filter_lab(lab_df, anchors, freq_threshold=FREQ_THRESHOLD):
    n0 = len(lab_df)
    lab_df = lab_df.merge(anchors[['person_id', 'index_date', 'timeframe_start']],
                          on='person_id', how='inner')
    mask = ((lab_df['measurement_datetime'] >= lab_df['timeframe_start']) &
            (lab_df['measurement_datetime'] < lab_df['index_date']))
    lab_df = lab_df[mask].copy()
    print(f"timeframe: {n0:,} → {len(lab_df):,}")

    total_persons = len(anchors)
    min_persons = int(total_persons * freq_threshold)
    person_counts = lab_df.groupby('standard_concept_code', observed=True)['person_id'].nunique()
    keep_codes = person_counts[person_counts >= min_persons].index
    lab_df = lab_df[lab_df['standard_concept_code'].isin(keep_codes)].copy()
    print(f"frequency filter (>={min_persons:,}人): "
          f"{lab_df['standard_concept_code'].nunique()} 个 code, {len(lab_df):,} 行")
    return lab_df


def clean_lab_values(lab_df):
    main_unit_map, rename_map, convert_map = build_unit_maps()
    df = lab_df.copy()

    df['unit_concept_name'] = df['unit_concept_name'].astype('object')
    df['standard_concept_code'] = df['standard_concept_code'].astype('object')

    missing_units = {'No matching concept', '<NULL>', 'no value', 'Null', None}
    df['unit_concept_name'] = df['unit_concept_name'].fillna('<NULL>')

    df['value_clean'] = df['value_as_number'].astype(float)
    df['unit_canonical'] = df['unit_concept_name'].copy()

    n_renamed = 0
    n_converted = 0
    n_dirty_nan = 0
    n_unknown_kept = 0

    for code in df['standard_concept_code'].unique():
        mask_code = df['standard_concept_code'] == code
        canonical = main_unit_map.get(code, 'unknown')
        rename_d = rename_map.get(code, {})
        convert_d = convert_map.get(code, {})

        if canonical == 'no_unit':
            df.loc[mask_code, 'unit_canonical'] = 'no_unit'
            continue

        sub_units = df.loc[mask_code, 'unit_concept_name'].unique()
        for u in sub_units:
            mask = mask_code & (df['unit_concept_name'] == u)

            if u == canonical:
                continue
            elif u in rename_d:
                df.loc[mask, 'unit_canonical'] = rename_d[u]
                n_renamed += mask.sum()
            elif u in convert_d:
                target, mult = convert_d[u]
                if mult == 'C_to_F':
                    df.loc[mask, 'value_clean'] = df.loc[mask, 'value_clean'] * 9/5 + 32
                else:
                    df.loc[mask, 'value_clean'] = df.loc[mask, 'value_clean'] * mult
                df.loc[mask, 'unit_canonical'] = target
                n_converted += mask.sum()
            elif u in missing_units:
                df.loc[mask, 'unit_canonical'] = canonical
                n_unknown_kept += mask.sum()
            else:
                df.loc[mask, 'value_clean'] = np.nan
                df.loc[mask, 'unit_canonical'] = 'DIRTY'
                n_dirty_nan += mask.sum()

    print(f"Unit cleaning done:")
    print(f"  rename:     {n_renamed:,} 行")
    print(f"  value convert:               {n_converted:,} 行")
    print(f"  unit NA:    {n_unknown_kept:,} 行")
    print(f"  illgal data → NaN:           {n_dirty_nan:,} 行")
    print(f"  value_clean exists:     {df['value_clean'].notna().mean()*100:.2f}%")

    return df


def build_lab_features(lab_df, anchors):
    all_codes = lab_df['standard_concept_code'].unique()
    numeric_codes = sorted([c for c in all_codes if c not in PURE_CONCEPT_CODES])
    concept_codes = sorted([c for c in all_codes
                            if c in PURE_CONCEPT_CODES and c != CBC_CODE])

    print(f"numeric lab: {len(numeric_codes)} 个")
    print(f"Concept lab: {len(concept_codes)} 个")
    print(f"CBC (onluy tested): 1 个")

    numeric_df = lab_df[lab_df['standard_concept_code'].isin(numeric_codes)].copy()

    count_df = (numeric_df.groupby(['person_id', 'standard_concept_code'], observed=True)
                .size().unstack(fill_value=0)
                .add_prefix('lab_count_'))
    print(f"  count matrix: {count_df.shape}")

    mean_df = (numeric_df.groupby(['person_id', 'standard_concept_code'], observed=True)['value_clean']
               .mean().unstack()
               .add_prefix('lab_mean_'))
    print(f"  mean matrix: {mean_df.shape}")

    numeric_df_sorted = numeric_df.sort_values(['person_id', 'standard_concept_code',
                                                 'measurement_datetime'])
    last_df = (numeric_df_sorted.dropna(subset=['value_clean'])
               .groupby(['person_id', 'standard_concept_code'], observed=True)['value_clean']
               .last().unstack()
               .add_prefix('lab_last_'))
    print(f"  last matrix: {last_df.shape}")

    concept_df = lab_df[lab_df['standard_concept_code'].isin(concept_codes)].copy()
    concept_df['_label'] = concept_df['value_as_concept_name'].apply(label_concept_value)

    tested_df = (concept_df.groupby(['person_id', 'standard_concept_code'], observed=True)
                 .size().unstack(fill_value=0)
                 .clip(upper=1)
                 .add_prefix('lab_tested_'))
    print(f"  tested matrix: {tested_df.shape}")

    pos_df = (concept_df[concept_df['_label']=='pos']
              .groupby(['person_id', 'standard_concept_code'], observed=True)
              .size().unstack(fill_value=0)
              .clip(upper=1)
              .add_prefix('lab_positive_'))
    for c in concept_codes:
        col = f'lab_positive_{c}'
        if col not in pos_df.columns:
            pos_df[col] = 0
    pos_df = pos_df[[f'lab_positive_{c}' for c in concept_codes]]
    print(f"  positive matrix: {pos_df.shape}")

    cbc_df = lab_df[lab_df['standard_concept_code'] == CBC_CODE]
    cbc_tested = (cbc_df.groupby('person_id').size()
                  .clip(upper=1).rename(f'lab_tested_{CBC_CODE}').to_frame())
    print(f"  CBC tested matrix: {cbc_tested.shape}")

    total_count = (lab_df.groupby('person_id').size()
                   .rename('lab_total_count').to_frame())
    unique_count = (lab_df.groupby('person_id')['standard_concept_code']
                    .nunique().rename('lab_unique_count').to_frame())

    base = anchors[['person_id', 'IsPositive']].drop_duplicates('person_id').set_index('person_id')

    feature_matrix = (base
                      .join(count_df, how='left')
                      .join(mean_df, how='left')
                      .join(last_df, how='left')
                      .join(tested_df, how='left')
                      .join(pos_df, how='left')
                      .join(cbc_tested, how='left')
                      .join(total_count, how='left')
                      .join(unique_count, how='left'))

    fill_zero_cols = [c for c in feature_matrix.columns
                      if c.startswith('lab_count_')
                      or c.startswith('lab_tested_')
                      or c.startswith('lab_positive_')
                      or c in ('lab_total_count', 'lab_unique_count')]
    feature_matrix[fill_zero_cols] = feature_matrix[fill_zero_cols].fillna(0).astype(np.int32)

    float_cols = [c for c in feature_matrix.columns
                  if c.startswith('lab_mean_') or c.startswith('lab_last_')]
    feature_matrix[float_cols] = feature_matrix[float_cols].astype(np.float32)

    feature_matrix = feature_matrix.reset_index()

    print(f"\nfinal feature matrix: {feature_matrix.shape}")
    print(f"  patients: {len(feature_matrix):,}")
    print(f"  feature ( person_id, IsPositive): {feature_matrix.shape[1]}")
    print(f"  pure feature: {feature_matrix.shape[1] - 2}")

    return feature_matrix


def run_lab_pipeline(timeframe, save=True, return_data=False):
    print(f"TIMEFRAME = {timeframe} months")

    anchors = load_anchors(timeframe)
    lab_df = load_lab_table(timeframe)
    lab_df = filter_lab(lab_df, anchors)

    lab_clean = clean_lab_values(lab_df)
    del lab_df
    gc.collect()

    lab_features = build_lab_features(lab_clean, anchors)

    if save:
        output_path = f"{feature_dir}/lab_features_{timeframe}.parquet"
        lab_features.to_parquet(output_path, index=False)
        print(f"saved: {output_path}")

    print("\n" + "="*60)
    print(f"Sanity check (TF={timeframe})")
    print("="*60)
    print(f"Shape: {lab_features.shape}")
    print(f"\nIsPositive distribution:")
    print(lab_features['IsPositive'].value_counts())
    print(f"\nkey feature non-zero:")
    key_cols = ['lab_total_count', 'lab_unique_count',
                'lab_tested_19295-5', 'lab_positive_19295-5',
                'lab_tested_3414-0', 'lab_positive_3414-0',
                'lab_tested_3773-9', 'lab_positive_3773-9']
    for col in key_cols:
        if col in lab_features.columns:
            nonzero = (lab_features[col] > 0).sum()
            pct = nonzero / len(lab_features) * 100
            print(f"  {col}: {nonzero:,} ({pct:.2f}%)")

    del anchors, lab_clean
    gc.collect()

    if return_data:
        return lab_features
    else:
        del lab_features
        gc.collect()
        return None




In [ ]:
for tf in [24]:
    run_lab_pipeline(tf)

In [ ]:
for tf in [6]:
    run_lab_pipeline(tf)

# Measurement

In [ ]:
import pandas as pd
import numpy as np
import re

BUCKET = 'fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf'
CLEAN_DIR = f'gs://{BUCKET}/amia/clean_data'
FEATURE_DIR = f'gs://{BUCKET}/amia/feature'

N_TOTAL = 267747
THR = 1338

LEAK_KW = ['opioid','opiate','narcotic','oxycodone','hydrocodone','fentanyl','morphine',
           'buprenorphine','methadone','naloxone','naltrexone','substance','addiction',
           'abuse','dependence','misuse','overdose','withdrawal','oud','sud',
           'drug screen','drug test','toxicology']

def clean_name(name, code, max_len=40):
    if pd.isna(name) or not str(name).strip():
        return f"meas_{str(code)[-6:]}"
    s = re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(name).lower())).strip('_')
    return (s[:max_len].rstrip('_') if len(s) > max_len else s) or f"meas_{str(code)[-6:]}"

def build_measurement_features(timeframe: int):
    print(f"\n{'='*60}")
    print(f"MEASUREMENT FEATURES — {timeframe} months")
    print(f"{'='*60}")

    TIME = 'measurement_datetime'
    CODE = 'standard_concept_code'
    NAME = 'standard_concept_name'

    m_pos = pd.read_csv(f'{CLEAN_DIR}/clean_measurement_{timeframe}.csv', low_memory=False)
    m_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_measurement_{timeframe}.csv', low_memory=False)
    print(f"[Load] pos: {m_pos.shape} | neg: {m_neg.shape}")

    m_pos[TIME] = pd.to_datetime(m_pos[TIME], errors='coerce').dt.normalize()
    m_neg[TIME] = pd.to_datetime(m_neg[TIME], errors='coerce').dt.normalize()

    all_cols = set(m_pos.columns) | set(m_neg.columns)
    for col in all_cols - set(m_pos.columns): m_pos[col] = pd.NA
    for col in all_cols - set(m_neg.columns): m_neg[col] = pd.NA
    table = pd.concat([m_pos[sorted(all_cols)], m_neg[sorted(all_cols)]], ignore_index=True)
    del m_pos, m_neg

    table[CODE] = table[CODE].astype(str)
    before = len(table)
    table = table.dropna(subset=[TIME]).copy()
    table['value_as_number'] = pd.to_numeric(table['value_as_number'], errors='coerce')
    print(f"[Clean] {before:,} → {len(table):,} (dropna time)")

    cp = (table.groupby([CODE, NAME])['person_id'].nunique()
          .reset_index(name='n_persons').sort_values('n_persons', ascending=False))
    passed = cp[cp['n_persons'] >= THR].copy()
    passed['leakage_flag'] = passed[NAME].astype(str).str.lower().apply(
        lambda x: any(k in x for k in LEAK_KW))
    print(f"[Filter] {len(cp)} → {len(passed)} concepts (leakage: {passed['leakage_flag'].sum()})")
    if passed['leakage_flag'].any():
        print(passed[passed['leakage_flag']][[CODE, NAME, 'n_persons']].to_string(index=False))

    kept_codes = passed[CODE].tolist()
    table = table[table[CODE].isin(kept_codes)].copy()

    code_type = table.groupby(CODE).apply(
        lambda g: 'numeric' if g['value_as_number'].notna().mean() >= 0.5 else 'categorical'
    ).to_dict()
    numeric_codes = [c for c, t in code_type.items() if t == 'numeric']
    categorical_codes = [c for c, t in code_type.items() if t == 'categorical']
    print(f"[Type] numeric: {len(numeric_codes)} | categorical: {len(categorical_codes)}")

    code_to_name = dict(zip(passed[CODE], passed[NAME]))
    base_names, seen = {}, set()
    for code in kept_codes:
        base = clean_name(code_to_name[code], code)
        if base in seen:
            base = f"{base}_{str(code)[-6:]}"
        seen.add(base)
        base_names[code] = base

    a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_{timeframe}.csv')
    a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{timeframe}.csv')
    a_pos['IsPositive'] = 1
    a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    assert len(anchors) == N_TOTAL, f"Anchor number error: {len(anchors)} != {N_TOTAL}"
    print(f"[Anchor] {len(anchors):,} ✓")

    CHUNK = 50  
    result = anchors[['person_id']].copy()

    def flush_frames(frames, result):
        for df in frames:
            result = result.merge(df, on='person_id', how='left')
        return result, []

    frames = []

    for i, code in enumerate(numeric_codes):
        sub = table[table[CODE] == code]
        b = base_names[code]
        cnt = sub.groupby('person_id').size().rename(f'meas_{b}_count')
        mean_ = sub.groupby('person_id')['value_as_number'].mean().rename(f'meas_{b}_mean')
        last = (sub.sort_values(TIME).dropna(subset=['value_as_number'])
                .groupby('person_id')['value_as_number'].last().rename(f'meas_{b}_last'))
        frames.append(pd.concat([cnt, mean_, last], axis=1).reset_index())
        if len(frames) >= CHUNK:
            result, frames = flush_frames(frames, result)
            print(f"  numeric chunk flushed @ {i+1}/{len(numeric_codes)}")

    for i, code in enumerate(categorical_codes):
        sub = table[table[CODE] == code]
        b = base_names[code]
        parts = [sub.groupby('person_id').size().rename(f'meas_{b}_count')]
        val_persons = sub.groupby('value_as_concept_name')['person_id'].nunique()
        for v in val_persons[val_persons >= THR].index:
            v_clean = clean_name(v, '000000', max_len=20)
            bin_col = (sub[sub['value_as_concept_name']==v].groupby('person_id').size()
                       .gt(0).astype('int8').rename(f'meas_{b}_{v_clean}'))
            parts.append(bin_col)
        frames.append(pd.concat(parts, axis=1).reset_index())
        if len(frames) >= CHUNK:
            result, frames = flush_frames(frames, result)
            print(f"  categorical chunk flushed @ {i+1}/{len(categorical_codes)}")

    frames.append(table.groupby('person_id').size().rename('meas_total_count').reset_index())
    frames.append(table.groupby('person_id')[CODE].nunique().rename('meas_unique_count').reset_index())
    result, frames = flush_frames(frames, result)

    for c in result.columns:
        if c == 'person_id': continue
        if c.endswith(('_mean', '_last')):
            result[c] = result[c].astype('float32')
        else:
            result[c] = result[c].fillna(0).astype('int32')

    print(f"[Result] shape: {result.shape}")

    check = result.merge(anchors, on='person_id', how='left')
    pm = check[check['IsPositive']==1]['meas_total_count'].mean()
    nm = check[check['IsPositive']==0]['meas_total_count'].mean()
    cov = (check['meas_total_count'] > 0).sum()
    print(f"[Sanity] pos={pm:.2f} | neg={nm:.2f} | ratio={pm/max(nm,1e-6):.2f}x | cov={cov/N_TOTAL*100:.1f}%")

    result.to_parquet(f'{FEATURE_DIR}/measurement_features_{timeframe}.parquet', index=False)
    nm_df = passed[[CODE, NAME, 'n_persons', 'leakage_flag']].copy()
    nm_df['column_base'] = nm_df[CODE].map(base_names)
    nm_df['type'] = nm_df[CODE].map(code_type)
    nm_df.to_csv(f'{FEATURE_DIR}/measurement_name_map_{timeframe}.csv', index=False)
    print(f"measurement_features_{timeframe}.parquet saved\n")

for tf in [6, 12, 24]:
    build_measurement_features(tf)

In [ ]:
for tf in [6]:
    build_measurement_features(tf)

In [ ]:
FEATURE_DIR = 'gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/feature'

for tf in [12, 24]:
    print(f"\n{'='*50}")
    print(f"measurement_features_{tf}.parquet")
    print(f"{'='*50}")
    df = pd.read_parquet(f'{FEATURE_DIR}/measurement_features_{tf}.parquet')
    print(f"shape: {df.shape}")
    print(f"columns: {df.columns.tolist()}")
    display(df.head(5))

# Drug

In [ ]:
import pandas as pd
import numpy as np
import re
import gc

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
clean_dir = f"{bucket}/amia/clean_data"
feature_dir = f"{bucket}/amia/feature"
FREQ_THRESHOLD = 0.005
N_ANCHOR_EXPECTED = 267747

DRUG_LEAKAGE_KEYWORDS = [
    'buprenorphine', 'methadone', 'naltrexone',
    'suboxone', 'subutex', 'vivitrol', 'sublocade',
    'naloxone', 'narcan',
]


def load_anchors(timeframe):
    pos_path = f"{clean_dir}/clean_positive_{timeframe}.csv"
    neg_path = f"{clean_dir}/clean_negative_anchor_{timeframe}.csv"
    
    print(f"Loading anchor: {pos_path.split('/')[-1]}")
    pos = pd.read_csv(pos_path)
    print(f"Loading anchor: {neg_path.split('/')[-1]}")
    neg = pd.read_csv(neg_path)
    
    required = {'person_id', 'index_date', 'timeframe_start', 'IsPositive'}
    assert required.issubset(pos.columns), f"pos lack column: {required - set(pos.columns)}"
    assert required.issubset(neg.columns), f"neg lack column: {required - set(neg.columns)}"
    
    pos = pos[['person_id', 'index_date', 'timeframe_start', 'IsPositive']]
    neg = neg[['person_id', 'index_date', 'timeframe_start', 'IsPositive']]
    
    for df in [pos, neg]:
        df['index_date'] = pd.to_datetime(df['index_date'], utc=True, errors='coerce').dt.tz_localize(None).dt.normalize()
        df['timeframe_start'] = pd.to_datetime(df['timeframe_start'], utc=True, errors='coerce').dt.tz_localize(None).dt.normalize()
    
    anchors = pd.concat([pos, neg], axis=0, ignore_index=True)

    assert len(anchors) == N_ANCHOR_EXPECTED, f"Anchor row number error: {len(anchors)} (expected {N_ANCHOR_EXPECTED})"
    assert anchors['person_id'].is_unique, "Anchor person_id repeated"
    assert anchors['index_date'].notna().all(), "with NaN index_date"
    assert anchors['timeframe_start'].notna().all(), "with NaN timeframe_start"
    assert (anchors['timeframe_start'] < anchors['index_date']).all(), "timeframe_start >= index_date error"
    
    print(f"Anchor @ {timeframe}m loaded: {len(anchors):,} rows "
          f"({(anchors['IsPositive']==1).sum():,} pos, {(anchors['IsPositive']==0).sum():,} neg)")
    return anchors


def load_drug_table(timeframe, chunk_size=500_000):
    cols = ['person_id', 'standard_concept_name', 'standard_concept_code',
            'drug_exposure_start_datetime']
    chunks = []
    for path in [f"{clean_dir}/clean_drug_{timeframe}.csv",
                 f"{clean_dir}/clean_negative_drug_{timeframe}.csv"]:
        print(f"Reading {path} ...")
        for chunk in pd.read_csv(path, usecols=cols, chunksize=chunk_size, low_memory=False):
            chunks.append(chunk)
    df = pd.concat(chunks, axis=0, ignore_index=True)
    print(f"Drug aggregate (raw): {len(df):,} rows")
    
    df['standard_concept_code'] = df['standard_concept_code'].astype(str)
    df['standard_concept_name'] = df['standard_concept_name'].astype(str)
    df['drug_exposure_start_datetime'] = (
        pd.to_datetime(df['drug_exposure_start_datetime'],
                       utc=True, errors='coerce')
        .dt.tz_localize(None).dt.normalize()
    )
    n_before = len(df)
    df = df.dropna(subset=['drug_exposure_start_datetime'])
    print(f"remove NAn: {n_before:,} → {len(df):,} ({len(df)/n_before*100:.1f}%)")
    return df


def filter_time_window_drug(drug_df, anchors):
    n_before = len(drug_df)
    drug_df = drug_df.merge(anchors[['person_id', 'index_date', 'timeframe_start']],
                            on='person_id', how='inner')
    n_merged = len(drug_df)
    print(f"Merge anchor: {n_before:,} → {n_merged:,}")
    
    mask = ((drug_df['drug_exposure_start_datetime'] >= drug_df['timeframe_start']) &
            (drug_df['drug_exposure_start_datetime'] < drug_df['index_date']))
    drug_df = drug_df[mask].copy()
    drug_df = drug_df.drop(columns=['index_date', 'timeframe_start'])
    print(f"timeframe filter: {n_merged:,} → {len(drug_df):,} ({len(drug_df)/n_merged*100:.1f}%)")
    return drug_df


def filter_low_freq_drug(drug_df, total_persons, threshold=FREQ_THRESHOLD):
    min_persons = int(total_persons * threshold)
    person_counts = drug_df.groupby('standard_concept_code')['person_id'].nunique()
    keep_codes = person_counts[person_counts >= min_persons].index
    n_before = drug_df['standard_concept_code'].nunique()
    drug_df = drug_df[drug_df['standard_concept_code'].isin(keep_codes)].copy()
    print(f"frequency filter ({min_persons:,} patients = {threshold*100}%):")
    print(f"  Code : {n_before:,} → {len(keep_codes):,}")
    print(f"  row number: → {len(drug_df):,}")
    return drug_df


def sanitize_column_name(name, max_len=80):
    name = re.sub(r'[/\\\s,;:()\[\]{}]+', '_', name)
    name = re.sub(r'[^\w-]', '', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_').lower()
    if len(name) > max_len:
        name = name[:max_len].rstrip('_')
    return name


def build_drug_features(drug_df, anchors):
    print("="*60)
    print("construct drug feature matrix")
    print("="*60)
    
    code_to_name = (drug_df.groupby('standard_concept_code')['standard_concept_name']
                    .agg(lambda x: x.mode().iloc[0]))
    
    code_to_colname = {}
    used_colnames = {}
    for code, name in code_to_name.items():
        clean = sanitize_column_name(name)
        if clean in used_colnames:
            clean = f"{clean}_{code[-6:]}"
        used_colnames[clean] = code
        code_to_colname[code] = clean
    
    print(f"Code column reflection: {len(code_to_colname)} ")
    print(f"\n top 10 reflection example:")
    for code, colname in list(code_to_colname.items())[:10]:
        name = code_to_name[code]
        name_short = name[:60] + '...' if len(name) > 60 else name
        print(f"  [{code}] {name_short}")
        print(f"     → drug_{colname}")
    
    feature_cols = {}
    all_codes = sorted(code_to_colname.keys())
    for i, code in enumerate(all_codes, 1):
        sub = drug_df[drug_df['standard_concept_code'] == code]
        cnt = sub.groupby('person_id').size()
        colname = f'drug_{code_to_colname[code]}'
        feature_cols[colname] = cnt.astype(np.int32)
        if i % 50 == 0 or i == len(all_codes):
            print(f"  [{i}/{len(all_codes)}]")
    
    feature_cols['drug_total_count'] = drug_df.groupby('person_id').size().astype(np.int32)
    feature_cols['drug_unique_count'] = (drug_df.groupby('person_id')['standard_concept_code']
                                         .nunique().astype(np.int32))
    
    base = (anchors[['person_id', 'IsPositive']]
            .drop_duplicates('person_id').set_index('person_id'))
    
    feat_df = pd.DataFrame(feature_cols)
    feature_matrix = base.join(feat_df, how='left')
    
    fill_cols = [c for c in feature_matrix.columns if c.startswith('drug_')]
    feature_matrix[fill_cols] = feature_matrix[fill_cols].fillna(0).astype(np.int32)
    
    feature_matrix = feature_matrix.reset_index()
    print(f"\nfinal feature matrix: {feature_matrix.shape}")
    
    name_map = pd.DataFrame([
        {'colname': f'drug_{code_to_colname[code]}',
         'standard_concept_code': code,
         'standard_concept_name': code_to_name[code],
         'is_leakage_candidate': any(
             kw in code_to_name[code].lower() for kw in DRUG_LEAKAGE_KEYWORDS
         )}
        for code in all_codes
    ])
    return feature_matrix, name_map


def run_drug_pipeline(timeframe):
    print(f"# Drug pipeline @ {timeframe} months")
    
    anchors = load_anchors(timeframe)
    drug_df = load_drug_table(timeframe)
    drug_df = filter_time_window_drug(drug_df, anchors)
    drug_df = filter_low_freq_drug(drug_df, total_persons=len(anchors))
    drug_features, drug_name_map = build_drug_features(drug_df, anchors)
    
    del drug_df
    gc.collect()
    
    output_path = f"{feature_dir}/drug_features_{timeframe}.parquet"
    drug_features.to_parquet(output_path, index=False)
    print(f"saved: {output_path}")
    
    name_map_path = f"{feature_dir}/drug_name_map_{timeframe}.csv"
    drug_name_map.to_csv(name_map_path, index=False)
    
    print(f"saved: {name_map_path}")
    print(f"Shape: {drug_features.shape}")
    print(f"\nIsPositive distribution:")
    print(drug_features['IsPositive'].value_counts())
    
    print(f"\npos vs neg:")
    for col in ['drug_total_count', 'drug_unique_count']:
        pos_mean = drug_features[drug_features['IsPositive']==1][col].mean()
        neg_mean = drug_features[drug_features['IsPositive']==0][col].mean()
        print(f"  {col}: pos_mean={pos_mean:.2f} | neg_mean={neg_mean:.2f}")
    
    leakage_cols = drug_name_map[drug_name_map['is_leakage_candidate']]['colname'].tolist()
    print(f"\n Leakage candidates flagged: {len(leakage_cols)}")
    for col in leakage_cols:
        if col in drug_features.columns:
            pos_rate = (drug_features[drug_features['IsPositive']==1][col] > 0).mean() * 100
            neg_rate = (drug_features[drug_features['IsPositive']==0][col] > 0).mean() * 100
            ratio = pos_rate / neg_rate if neg_rate > 0 else float('inf')
            print(f"  {col}: pos={pos_rate:.2f}% | neg={neg_rate:.2f}% | ratio={ratio:.1f}x")
    
    has_drug = (drug_features['drug_total_count'] > 0).sum()
    print(f"\n drug record: {has_drug:,} / {len(drug_features):,} "
          f"({has_drug/len(drug_features)*100:.1f}%)")
    
    return drug_features, drug_name_map

In [ ]:
# 12m
drug_features_12, drug_name_map_12 = run_drug_pipeline(timeframe=12)

del drug_features_12, drug_name_map_12
gc.collect()
drug_features_24, drug_name_map_24 = run_drug_pipeline(timeframe=24)

# Observation

In [ ]:
import pandas as pd
import numpy as np
import re

DEFAULT_BUCKET = 'fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf'
DEFAULT_POSTAL = '45401-7'

DEFAULT_LEAK_KW = [
    'opioid', 'opiate', 'narcotic', 'oxycodone', 'hydrocodone', 'fentanyl', 'morphine',
    'buprenorphine', 'methadone', 'naloxone', 'naltrexone', 'substance', 'addiction',
    'abuse', 'dependence', 'misuse', 'overdose', 'withdrawal', 'oud', 'sud',
    'pain medication', 'pain management', 'chronic pain', 'drug screen', 'drug test',
    'toxicology', 'tobacco', 'smoking', 'alcohol'
]

TIME = 'observation_datetime'
CODE = 'standard_concept_code'
NAME = 'standard_concept_name'

def clean_name(name, code, max_len=40):
    if pd.isna(name) or not str(name).strip():
        return f"obs_{str(code)[-6:]}"
    s = re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(name).lower())).strip('_')
    return (s[:max_len].rstrip('_') if len(s) > max_len else s) or f"obs_{str(code)[-6:]}"

def build_observation_features(
    timeframe,
    n_total,
    threshold,
    bucket=DEFAULT_BUCKET,
    postal_code=DEFAULT_POSTAL,
    leak_keywords=None,
    save=True,
    verbose=True,
):
    if leak_keywords is None:
        leak_keywords = DEFAULT_LEAK_KW

    clean_dir = f'gs://{bucket}/amia/clean_data'
    feature_dir = f'gs://{bucket}/amia/feature'

    log = print if verbose else (lambda *a, **k: None)
    log(f"\n{'='*60}\n[Pipeline] timeframe={timeframe} | n_total={n_total} | threshold={threshold}\n{'='*60}")

    o_pos = pd.read_csv(f'{clean_dir}/clean_observation_{timeframe}.csv', low_memory=False)
    o_neg = pd.read_csv(f'{clean_dir}/clean_negative_observation_{timeframe}.csv', low_memory=False)
    log(f"[Load] pos: {o_pos.shape} | neg: {o_neg.shape}")

    postal_persons = set(pd.concat([
        o_pos.loc[o_pos[CODE].astype(str) == postal_code, 'person_id'],
        o_neg.loc[o_neg[CODE].astype(str) == postal_code, 'person_id'],
    ]).unique())
    o_pos = o_pos[o_pos[CODE].astype(str) != postal_code]
    o_neg = o_neg[o_neg[CODE].astype(str) != postal_code]
    log(f"[Postal] {len(postal_persons)} 人")

    o_pos[TIME] = pd.to_datetime(o_pos[TIME], errors='coerce').dt.normalize()
    o_neg[TIME] = pd.to_datetime(o_neg[TIME], errors='coerce').dt.normalize()
    all_cols = set(o_pos.columns) | set(o_neg.columns)
    for col in all_cols - set(o_pos.columns):
        o_pos[col] = pd.NA
    for col in all_cols - set(o_neg.columns):
        o_neg[col] = pd.NA
    table = pd.concat(
        [o_pos[sorted(all_cols)], o_neg[sorted(all_cols)]],
        ignore_index=True
    )
    del o_pos, o_neg

    table[CODE] = table[CODE].astype(str)
    before = len(table)
    table = table.dropna(subset=[TIME]).copy()
    table['value_as_number'] = pd.to_numeric(table['value_as_number'], errors='coerce')
    log(f"[Clean] dropna 时间: {before} → {len(table)}")

    cp = (table.groupby([CODE, NAME])['person_id'].nunique()
          .reset_index(name='n_persons').sort_values('n_persons', ascending=False))
    passed = cp[cp['n_persons'] >= threshold].copy()
    passed['leakage_flag'] = passed[NAME].astype(str).str.lower().apply(
        lambda x: any(k in x for k in leak_keywords))
    log(f"[Filter] {len(cp)} → {len(passed)} (leakage: {passed['leakage_flag'].sum()})")
    if verbose:
        log(passed[[CODE, NAME, 'n_persons', 'leakage_flag']].to_string(index=False))

    kept_codes = passed[CODE].tolist()
    table = table[table[CODE].isin(kept_codes)].copy()

    code_type = table.groupby(CODE).apply(
        lambda g: 'numeric' if g['value_as_number'].notna().mean() >= 0.5 else 'categorical'
    ).to_dict()
    numeric_codes = [c for c, t in code_type.items() if t == 'numeric']
    categorical_codes = [c for c, t in code_type.items() if t == 'categorical']
    log(f"[Type] numeric: {len(numeric_codes)} | categorical: {len(categorical_codes)}")

    code_to_name = dict(zip(passed[CODE], passed[NAME]))
    base_names, seen = {}, set()
    for code in kept_codes:
        base = clean_name(code_to_name[code], code)
        if base in seen:
            base = f"{base}_{str(code)[-6:]}"
        seen.add(base)
        base_names[code] = base

    a_pos = pd.read_csv(f'{clean_dir}/clean_positive_{timeframe}.csv')
    a_neg = pd.read_csv(f'{clean_dir}/clean_negative_anchor_{timeframe}.csv')
    a_pos['IsPositive'] = 1
    a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    log(f"  Anchor: {len(anchors):,}")
    assert len(anchors) == n_total, f"Anchor number error: {len(anchors)} != {n_total}"

    # -------- Aggregate --------
    frames = []
    for code in numeric_codes:
        sub = table[table[CODE] == code]
        b = base_names[code]
        cnt = sub.groupby('person_id').size().rename(f'obs_{b}_count')
        mean = sub.groupby('person_id')['value_as_number'].mean().rename(f'obs_{b}_mean')
        last = (sub.sort_values(TIME).dropna(subset=['value_as_number'])
                .groupby('person_id')['value_as_number'].last().rename(f'obs_{b}_last'))
        frames.append(pd.concat([cnt, mean, last], axis=1).reset_index())

    for code in categorical_codes:
        sub = table[table[CODE] == code]
        b = base_names[code]
        parts = [sub.groupby('person_id').size().rename(f'obs_{b}_count')]
        val_persons = sub.groupby('value_as_concept_name')['person_id'].nunique()
        for v in val_persons[val_persons >= threshold].index:
            v_clean = clean_name(v, '000000', max_len=20)
            bin_col = (sub[sub['value_as_concept_name'] == v].groupby('person_id').size()
                       .gt(0).astype('int8').rename(f'obs_{b}_{v_clean}'))
            parts.append(bin_col)
        frames.append(pd.concat(parts, axis=1).reset_index())

    frames.append(table.groupby('person_id').size().rename('obs_total_count').reset_index())
    frames.append(table.groupby('person_id')[CODE].nunique().rename('obs_unique_count').reset_index())

    result = anchors[['person_id']].copy()
    for df in frames:
        result = result.merge(df, on='person_id', how='left')
    result['obs_has_postal_code'] = result['person_id'].isin(postal_persons).astype('int8')

    for c in result.columns:
        if c == 'person_id' or c == 'obs_has_postal_code':
            continue
        if c.endswith(('_mean', '_last')):
            result[c] = result[c].astype('float32')
        else:
            result[c] = result[c].fillna(0).astype('int32')

    log(f"[Result] shape: {result.shape}")

    check = result.merge(anchors, on='person_id', how='left')
    log(f"\n[Sanity]")
    log(f"IsPositive distribution:\n{check['IsPositive'].value_counts()}")
    for col in ['obs_total_count', 'obs_unique_count']:
        pm = check[check['IsPositive'] == 1][col].mean()
        nm = check[check['IsPositive'] == 0][col].mean()
        log(f"  {col}: pos={pm:.2f} | neg={nm:.2f} | ratio={pm/max(nm,1e-6):.2f}x")
    cov = (check['obs_total_count'] > 0).sum()
    log(f" cover rate: {cov:,}/{n_total} ({cov/n_total*100:.1f}%)")

    if save:
        # 1) feature matrix
        result.to_parquet(
            f'{feature_dir}/observation_features_{timeframe}.parquet',
            index=False
        )

        # 2) name map
        nm_df = passed[[CODE, NAME, 'n_persons', 'leakage_flag']].copy()
        nm_df['column_base'] = nm_df[CODE].map(base_names)
        nm_df['type'] = nm_df[CODE].map(code_type)
        nm_df = pd.concat([nm_df, pd.DataFrame([{
            CODE: postal_code,
            NAME: 'Postal code [Location]',
            'n_persons': len(postal_persons),
            'leakage_flag': False,
            'column_base': 'has_postal_code',
            'type': 'static_binary',
        }])], ignore_index=True)
        nm_df.to_csv(
            f'{feature_dir}/observation_name_map_{timeframe}.csv',
            index=False
        )
        log(f"[Save] saved to {feature_dir}/observation_features_{timeframe}.parquet")
        log(f"[Save] saved to {feature_dir}/observation_name_map_{timeframe}.csv")

    log(f"[Done] timeframe={timeframe}\n")
    return result, passed


def build_all_timeframes(timeframes=(6, 12, 24), n_total=267747, threshold=1338, **kwargs):
    results = {}
    for tf in timeframes:
        result, passed = build_observation_features(
            timeframe=tf, n_total=n_total, threshold=threshold, **kwargs
        )
        results[tf] = {'result': result, 'passed': passed}
    return results

if __name__ == '__main__':
    all_results = build_all_timeframes(
        timeframes=(6, 12, 24),
        n_total=267747,
        threshold=1338,
    )

In [ ]:
import pandas as pd
import numpy as np
import re

BUCKET = 'fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf'
CLEAN_DIR = f'gs://{BUCKET}/amia/clean_data'
FEATURE_DIR = f'gs://{BUCKET}/amia/feature'
POSTAL = '45401-7'
N_TOTAL = 267747

TIME = 'observation_datetime'
CODE = 'standard_concept_code'
NAME = 'standard_concept_name'

LEAK_KW = ['opioid', 'opiate', 'narcotic', 'oxycodone', 'hydrocodone', 'fentanyl', 'morphine',
           'buprenorphine', 'methadone', 'naloxone', 'naltrexone', 'substance', 'addiction',
           'abuse', 'dependence', 'misuse', 'overdose', 'withdrawal', 'oud', 'sud',
           'pain medication', 'pain management', 'chronic pain', 'drug screen', 'drug test',
           'toxicology', 'tobacco', 'smoking', 'alcohol']


def clean_name(name, code, max_len=40):
    if pd.isna(name) or not str(name).strip():
        return f"obs_{str(code)[-6:]}"
    s = re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(name).lower())).strip('_')
    return (s[:max_len].rstrip('_') if len(s) > max_len else s) or f"obs_{str(code)[-6:]}"


def build_observation_features(
    timeframe,
    threshold=None,
    concept_whitelist=None,
    categorical_value_whitelist=None,
    save=True,
    verbose=True,
):
    if concept_whitelist is None and threshold is None:
        raise ValueError("require threshold or concept_whitelist")

    log = print if verbose else (lambda *a, **k: None)
    mode = 'whitelist' if concept_whitelist is not None else f'threshold={threshold}'
    log(f"[Pipeline] timeframe={timeframe} | mode={mode}")

    o_pos = pd.read_csv(f'{CLEAN_DIR}/clean_observation_{timeframe}.csv', low_memory=False)
    o_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_observation_{timeframe}.csv', low_memory=False)
    log(f"[Load] pos: {o_pos.shape} | neg: {o_neg.shape}")

    postal_persons = set(pd.concat([
        o_pos.loc[o_pos[CODE].astype(str) == POSTAL, 'person_id'],
        o_neg.loc[o_neg[CODE].astype(str) == POSTAL, 'person_id'],
    ]).unique())
    o_pos = o_pos[o_pos[CODE].astype(str) != POSTAL]
    o_neg = o_neg[o_neg[CODE].astype(str) != POSTAL]
    log(f"[Postal] {len(postal_persons)} persons")

    o_pos[TIME] = pd.to_datetime(o_pos[TIME], errors='coerce').dt.normalize()
    o_neg[TIME] = pd.to_datetime(o_neg[TIME], errors='coerce').dt.normalize()
    all_cols = set(o_pos.columns) | set(o_neg.columns)
    for col in all_cols - set(o_pos.columns):
        o_pos[col] = pd.NA
    for col in all_cols - set(o_neg.columns):
        o_neg[col] = pd.NA
    table = pd.concat([o_pos[sorted(all_cols)], o_neg[sorted(all_cols)]], ignore_index=True)
    del o_pos, o_neg

    table[CODE] = table[CODE].astype(str)
    before = len(table)
    table = table.dropna(subset=[TIME]).copy()
    table['value_as_number'] = pd.to_numeric(table['value_as_number'], errors='coerce')
    log(f"[Clean] dropna: {before} -> {len(table)}")

    cp = (table.groupby([CODE, NAME])['person_id'].nunique()
          .reset_index(name='n_persons').sort_values('n_persons', ascending=False))

    if concept_whitelist is not None:
        wl = set(str(c) for c in concept_whitelist)
        passed = cp[cp[CODE].isin(wl)].copy()
        missing = wl - set(cp[CODE])
        if missing:
            log(f"[Whitelist] code not in timeframe={timeframe}: {missing}")
        log(f"[Filter] whitelist: {len(wl)}, passed {len(passed)}")
    else:
        passed = cp[cp['n_persons'] >= threshold].copy()
        log(f"[Filter] {len(cp)} -> {len(passed)} (>={threshold})")

    passed['leakage_flag'] = passed[NAME].astype(str).str.lower().apply(
        lambda x: any(k in x for k in LEAK_KW))
    log(f"[Leakage] {passed['leakage_flag'].sum()}")
    if verbose:
        log(passed[[CODE, NAME, 'n_persons', 'leakage_flag']].to_string(index=False))

    kept_codes = passed[CODE].tolist()
    table = table[table[CODE].isin(kept_codes)].copy()

    code_type = table.groupby(CODE).apply(
        lambda g: 'numeric' if g['value_as_number'].notna().mean() >= 0.5 else 'categorical'
    ).to_dict()
    numeric_codes = [c for c, t in code_type.items() if t == 'numeric']
    categorical_codes = [c for c, t in code_type.items() if t == 'categorical']
    log(f"[Type] numeric: {len(numeric_codes)} | categorical: {len(categorical_codes)}")

    code_to_name = dict(zip(passed[CODE], passed[NAME]))
    base_names, seen = {}, set()
    for code in kept_codes:
        base = clean_name(code_to_name[code], code)
        if base in seen:
            base = f"{base}_{str(code)[-6:]}"
        seen.add(base)
        base_names[code] = base

    a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_{timeframe}.csv')
    a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{timeframe}.csv')
    a_pos['IsPositive'] = 1
    a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    log(f"[Anchor] {len(anchors):,}")

    frames = []
    cat_values_used = {}

    for code in numeric_codes:
        sub = table[table[CODE] == code]
        b = base_names[code]
        cnt = sub.groupby('person_id').size().rename(f'obs_{b}_count')
        mean = sub.groupby('person_id')['value_as_number'].mean().rename(f'obs_{b}_mean')
        last = (sub.sort_values(TIME).dropna(subset=['value_as_number'])
                .groupby('person_id')['value_as_number'].last().rename(f'obs_{b}_last'))
        frames.append(pd.concat([cnt, mean, last], axis=1).reset_index())

    for code in categorical_codes:
        sub = table[table[CODE] == code]
        b = base_names[code]
        parts = [sub.groupby('person_id').size().rename(f'obs_{b}_count')]

        if categorical_value_whitelist is not None and code in categorical_value_whitelist:
            target_values = categorical_value_whitelist[code]
        else:
            val_persons = sub.groupby('value_as_concept_name')['person_id'].nunique()
            v_thr = threshold if threshold is not None else 1
            target_values = val_persons[val_persons >= v_thr].index.tolist()

        used_values = []
        for v in target_values:
            v_clean = clean_name(v, '000000', max_len=20)
            mask = sub['value_as_concept_name'] == v
            if mask.any():
                bin_col = (sub[mask].groupby('person_id').size()
                           .gt(0).astype('int8').rename(f'obs_{b}_{v_clean}'))
                parts.append(bin_col)
            used_values.append(v)
        cat_values_used[code] = used_values
        frames.append(pd.concat(parts, axis=1).reset_index())

    frames.append(table.groupby('person_id').size().rename('obs_total_count').reset_index())
    frames.append(table.groupby('person_id')[CODE].nunique().rename('obs_unique_count').reset_index())

    result = anchors[['person_id']].copy()
    for df in frames:
        result = result.merge(df, on='person_id', how='left')
    result['obs_has_postal_code'] = result['person_id'].isin(postal_persons).astype('int8')

    for c in result.columns:
        if c == 'person_id' or c == 'obs_has_postal_code':
            continue
        if c.endswith(('_mean', '_last')):
            result[c] = result[c].astype('float32')
        else:
            result[c] = result[c].fillna(0).astype('int32')

    log(f"[Result] shape: {result.shape}")

    metadata = {
        'passed': passed,
        'base_names': base_names,
        'code_type': code_type,
        'categorical_values': cat_values_used,
        'kept_codes': kept_codes,
    }

    if save:
        result.to_parquet(f'{FEATURE_DIR}/observation_features_{timeframe}.parquet', index=False)
        nm_df = passed[[CODE, NAME, 'n_persons', 'leakage_flag']].copy()
        nm_df['column_base'] = nm_df[CODE].map(base_names)
        nm_df['type'] = nm_df[CODE].map(code_type)
        nm_df = pd.concat([nm_df, pd.DataFrame([{
            CODE: POSTAL, NAME: 'Postal code [Location]',
            'n_persons': len(postal_persons), 'leakage_flag': False,
            'column_base': 'has_postal_code', 'type': 'static_binary',
        }])], ignore_index=True)
        nm_df.to_csv(f'{FEATURE_DIR}/observation_name_map_{timeframe}.csv', index=False)
        log(f"[Save] features_{timeframe}.parquet & name_map_{timeframe}.csv")

    log(f"[Done] timeframe={timeframe}")
    return result, metadata


def build_aligned_timeframes(timeframes=(6, 12, 24), anchor_timeframe=6,
                             threshold=1338, save=True, verbose=True):
    log = print if verbose else (lambda *a, **k: None)
    log(f"[Aligned] anchor={anchor_timeframe} | tfs={timeframes}")

    log(f"[Step 1] build baseline with timeframe={anchor_timeframe}")
    anchor_result, anchor_meta = build_observation_features(
        timeframe=anchor_timeframe, threshold=threshold,
        save=save, verbose=verbose,
    )

    base_codes = anchor_meta['kept_codes']
    base_cat_values = anchor_meta['categorical_values']
    log(f"[Baseline] {len(base_codes)} concepts: {base_codes}")

    results = {anchor_timeframe: {'result': anchor_result, 'metadata': anchor_meta}}

    for tf in timeframes:
        if tf == anchor_timeframe:
            continue
        log(f"[Step 2] lock timeframe={tf}")
        result, meta = build_observation_features(
            timeframe=tf, concept_whitelist=base_codes,
            categorical_value_whitelist=base_cat_values,
            save=save, verbose=verbose,
        )
        results[tf] = {'result': result, 'metadata': meta}

    cols_by_tf = {tf: set(results[tf]['result'].columns) for tf in timeframes}
    common = set.intersection(*cols_by_tf.values())
    union = set.union(*cols_by_tf.values())
    log(f"[Align] total: {len(union)} | common: {len(common)}")
    for tf in timeframes:
        diff = cols_by_tf[tf] - common
        log(f"  tf={tf}: shape={results[tf]['result'].shape}, unique {len(diff)}: {diff if diff else 'none'}")

    return results


all_results = build_aligned_timeframes(
    timeframes=(6, 12, 24),
    anchor_timeframe=6,
    threshold=1338,
    save=False,
)

In [ ]:
import pandas as pd
CLEAN_DIR = 'gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data'
a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_6.csv')[['person_id']]
a_pos['IsPositive'] = 1
a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_6.csv')[['person_id']]
a_neg['IsPositive'] = 0
labels = pd.concat([a_pos, a_neg], ignore_index=True)

print(f"{'tf':<5} {'group':<10} {'mean_total':>12} {'median':>10} {'max':>10} {'>0 account':>10}")
print('-' * 60)
for tf in [6, 12, 24]:
    res = all_results[tf]['result']
    m = res.merge(labels, on='person_id', how='left')
    for grp, lbl in [(1, 'pos'), (0, 'neg')]:
        sub = m[m['IsPositive'] == grp]['obs_total_count']
        print(f"{tf:<5} {lbl:<10} {sub.mean():>12.3f} {sub.median():>10.0f} {sub.max():>10.0f} {(sub>0).mean()*100:>9.1f}%")
    print()

# Condition

In [ ]:
import pandas as pd
import numpy as np
import gc

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
clean_dir = f"{BASE}/amia/clean_data"
feature_dir = f"{BASE}/amia/feature"

FREQ_THRESHOLD = 0.005

def load_anchors(timeframe):
    pos = pd.read_csv(f"{clean_dir}/clean_positive_{timeframe}.csv",
                      usecols=['person_id', 'index_date', 'timeframe_start'])
    neg = pd.read_csv(f"{clean_dir}/clean_negative_anchor_{timeframe}.csv",
                      usecols=['person_id', 'index_date', 'timeframe_start'])
    pos['IsPositive'] = 1
    neg['IsPositive'] = 0
    anchors = pd.concat([pos, neg], axis=0, ignore_index=True)
    anchors['index_date'] = pd.to_datetime(anchors['index_date'], utc=True).dt.tz_localize(None).dt.normalize()
    anchors['timeframe_start'] = pd.to_datetime(anchors['timeframe_start'], utc=True).dt.tz_localize(None).dt.normalize()
    print(f"[{timeframe}mo] Anchor: {len(anchors):,} 人 (positive={pos.shape[0]:,}, negative={neg.shape[0]:,})")
    return anchors

def load_condition_table(timeframe, chunk_size=500_000):
    cols = ['person_id', 'standard_concept_code', 'condition_start_datetime']
    chunks = []
    for path in [f"{clean_dir}/clean_condition_{timeframe}.csv",
                 f"{clean_dir}/clean_negative_condition_{timeframe}.csv"]:
        print(f"Reading {path} ...")
        for chunk in pd.read_csv(path, usecols=cols, chunksize=chunk_size, low_memory=False):
            chunks.append(chunk)
    df = pd.concat(chunks, axis=0, ignore_index=True)
    print(f"[{timeframe}mo] Condition aggregate (raw): {len(df):,} 行")
    
    d = df['condition_start_datetime'].astype(str).str.slice(0, 10)
    df['condition_start_datetime'] = pd.to_datetime(d, format='%Y-%m-%d', errors='coerce')

    n_before = len(df)
    df = df.dropna(subset=['condition_start_datetime'])
    print(f"[{timeframe}mo] remove NaN time: {n_before:,} → {len(df):,} ({len(df)/n_before*100:.1f}%)")
    df['standard_concept_code'] = df['standard_concept_code'].astype(str)
    return df

def filter_time_window_cond(cond_df, anchors, timeframe):
    n_before = len(cond_df)
    cond_df = cond_df.merge(anchors[['person_id', 'index_date', 'timeframe_start']],
                            on='person_id', how='inner')
    n_merged = len(cond_df)
    print(f"[{timeframe}mo] Merge anchor: {n_before:,} → {n_merged:,}")

    mask = ((cond_df['condition_start_datetime'] >= cond_df['timeframe_start']) &
            (cond_df['condition_start_datetime'] < cond_df['index_date']))
    cond_df = cond_df[mask].copy()
    cond_df = cond_df.drop(columns=['index_date', 'timeframe_start'])
    print(f"[{timeframe}mo] timeframe filter: {n_merged:,} → {len(cond_df):,} ({len(cond_df)/n_merged*100:.1f}%)")
    return cond_df

def filter_low_freq_cond(cond_df, total_persons, timeframe, threshold=FREQ_THRESHOLD):
    min_persons = int(total_persons * threshold)
    person_counts = cond_df.groupby('standard_concept_code')['person_id'].nunique()
    keep_codes = person_counts[person_counts >= min_persons].index
    n_before = cond_df['standard_concept_code'].nunique()
    cond_df = cond_df[cond_df['standard_concept_code'].isin(keep_codes)].copy()
    print(f"[{timeframe}mo] frequency filter ( {min_persons:,}  = {threshold*100}%):")
    print(f"  Code : {n_before:,} → {len(keep_codes):,}")
    print(f"  rows: → {len(cond_df):,}")
    return cond_df

def build_condition_features(cond_df, anchors, timeframe):
    print("="*60)
    print(f"[{timeframe}mo] construct condition feature matrix")
    print("="*60)

    all_codes = sorted(cond_df['standard_concept_code'].unique())
    print(f"Code: {len(all_codes)}")

    feature_cols = {}
    for i, code in enumerate(all_codes, 1):
        sub = cond_df[cond_df['standard_concept_code'] == code]
        cnt = sub.groupby('person_id').size()
        feature_cols[f'cond_count_{code}'] = cnt.astype(np.int32)
        if i % 100 == 0 or i == len(all_codes):
            print(f"  [{i}/{len(all_codes)}]")

    feature_cols['cond_total_count'] = cond_df.groupby('person_id').size().astype(np.int32)
    feature_cols['cond_unique_count'] = (cond_df.groupby('person_id')['standard_concept_code']
                                         .nunique().astype(np.int32))
    base = (anchors[['person_id', 'IsPositive']]
            .drop_duplicates('person_id').set_index('person_id'))

    feat_df = pd.DataFrame(feature_cols)
    feature_matrix = base.join(feat_df, how='left')

    fill_cols = [c for c in feature_matrix.columns if c.startswith('cond_')]
    feature_matrix[fill_cols] = feature_matrix[fill_cols].fillna(0).astype(np.int32)
    feature_matrix = feature_matrix.reset_index()
    print(f"[{timeframe}mo] final feature matrix: {feature_matrix.shape}")
    return feature_matrix

def run_pipeline(timeframe):
    anchors = load_anchors(timeframe)
    cond_df = load_condition_table(timeframe)
    cond_df = filter_time_window_cond(cond_df, anchors, timeframe)
    cond_df = filter_low_freq_cond(cond_df, total_persons=len(anchors), timeframe=timeframe)
    cond_features = build_condition_features(cond_df, anchors, timeframe)
    del cond_df
    gc.collect()

    output_path = f"{feature_dir}/cond_features_{timeframe}.parquet"
    cond_features.to_parquet(output_path, index=False)
    print(f"saved: {output_path}")
    print(f"Shape: {cond_features.shape}")
    print(f"\nIsPositive distribution:\n{cond_features['IsPositive'].value_counts()}")

    for col in ['cond_total_count', 'cond_unique_count']:
        pos_mean = cond_features[cond_features['IsPositive']==1][col].mean()
        neg_mean = cond_features[cond_features['IsPositive']==0][col].mean()
        print(f"  {col}: pos_mean={pos_mean:.2f} | neg_mean={neg_mean:.2f}")
    has_cond = (cond_features['cond_total_count'] > 0).sum()
    print(f"\npatients with condition: {has_cond:,} / {len(cond_features):,} ({has_cond/len(cond_features)*100:.1f}%)")

    return cond_features

for tf in [6,12,24]:
    result = run_pipeline(tf)
    del result
    gc.collect()

In [ ]:
cf = pd.read_parquet(f"{feature_dir}/cond_features_6.parquet")
cond_cols = [c for c in cf.columns if c.startswith('cond_count_')]
neg = cf[cf['IsPositive']==0]
print("shape:", cf.shape, " cond_count column number:", len(cond_cols),
      " negative all 0 ratio:", (neg[cond_cols].sum(axis=1)==0).mean())

# Survey

In [ ]:
import pandas as pd
import numpy as np
import re

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR = f'{BASE}/amia/clean_data'
FEATURE_DIR = f'{BASE}/amia/feature'

N_TOTAL = 267747
RESPONSE_THR = 0.50

def build_survey_features(timeframe: int):
    print(f"[Survey features] {timeframe} months")

    TIME = 'survey_datetime'
    s_pos = pd.read_csv(f'{CLEAN_DIR}/clean_survey_{timeframe}.csv', low_memory=False)
    s_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_survey_{timeframe}.csv', low_memory=False)
    print(f"[Load] pos: {s_pos.shape} | neg: {s_neg.shape}")

    s_pos[TIME] = pd.to_datetime(s_pos[TIME], errors='coerce').dt.normalize()
    s_neg[TIME] = pd.to_datetime(s_neg[TIME], errors='coerce').dt.normalize()

    all_cols = set(s_pos.columns) | set(s_neg.columns)
    for col in all_cols - set(s_pos.columns): s_pos[col] = pd.NA
    for col in all_cols - set(s_neg.columns): s_neg[col] = pd.NA
    table = pd.concat([s_pos[sorted(all_cols)], s_neg[sorted(all_cols)]], ignore_index=True)
    del s_pos, s_neg

    before = len(table)
    table = table.dropna(subset=[TIME]).copy()
    print(f"[Clean] dropna time: {before:,} → {len(table):,}")

    for c in ['question_concept_id', 'answer_concept_id']:
        table[c] = pd.to_numeric(table[c], errors='coerce')
    n_bad_id = table['question_concept_id'].isna().sum()
    if n_bad_id:
        print(f"[WARN] question_concept_id missing in {n_bad_id} rows, dropped")
        table = table.dropna(subset=['question_concept_id']).copy()
    table['question_concept_id'] = table['question_concept_id'].astype('int64')

    is_skip = table['answer'].astype(str).str.strip().eq('PMI: Skip')
    skipped = table[is_skip].copy()
    answered = table[~is_skip].copy()
    print(f"[Split] answered: {len(answered):,} | Skip: {len(skipped):,}")
    
    participants = set(answered['person_id'].unique())
    n_participants = len(participants)
    print(f"[Participants] answered >=1 concrete question: {n_participants:,}")


    q_response = (answered.groupby(['question_concept_id', 'question'])['person_id']
                  .nunique().reset_index(name='n_answered'))
    q_response['response_rate'] = q_response['n_answered'] / n_participants
    print(f"[Question response-rate distribution]")
    print(f"  total questions: {len(q_response)}")
    print(f"  response rate ≥ 50%: {(q_response['response_rate'] >= RESPONSE_THR).sum()}")

    kept_qids = q_response[q_response['response_rate'] >= RESPONSE_THR]['question_concept_id'].tolist()
    print(f"\n[Filter] kept {len(kept_qids)} questions (response rate ≥ {RESPONSE_THR*100:.0f}%)")

    answered = answered[answered['question_concept_id'].isin(kept_qids)].copy()
    skipped = skipped[skipped['question_concept_id'].isin(kept_qids)].copy()

    ans_bad = answered['answer_concept_id'].isna() | (pd.to_numeric(answered['answer_concept_id'], errors='coerce').isna())
    if ans_bad.any():
        print("[Diagnostic] rows still missing answer_concept_id after filter, distribution by question:")
        print(answered[ans_bad]['question'].value_counts().head(10))

    answered_valid = answered.copy()
    answered_valid['answer_concept_id'] = pd.to_numeric(answered_valid['answer_concept_id'], errors='coerce')
    n_bad_ans = answered_valid['answer_concept_id'].isna().sum()
    if n_bad_ans:
        print(f"dropped {n_bad_ans} rows missing answer_concept_id when building binary columns"
              f" (still counted in surv_total_count / surv_unique_question_count, only without a specific answer column)")
        answered_valid = answered_valid.dropna(subset=['answer_concept_id']).copy()
    answered_valid['answer_concept_id'] = answered_valid['answer_concept_id'].astype('int64')

    answered_valid['qa_key'] = ('surv_q' + answered_valid['question_concept_id'].astype(str)
                                 + '__a' + answered_valid['answer_concept_id'].astype(str))
    qa_keys = sorted(answered_valid['qa_key'].unique())
    print(f"[qa_keys] total binary columns: {len(qa_keys)}")
    key_to_qid = answered_valid.groupby('qa_key')['question_concept_id'].first().to_dict()
    key_to_persons = answered_valid.groupby('qa_key')['person_id'].apply(set).to_dict()

    a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_{timeframe}.csv')
    a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{timeframe}.csv')
    a_pos['IsPositive'] = 1
    a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    assert len(anchors) == N_TOTAL, f"Anchor mismatch: {len(anchors)} != {N_TOTAL}"
    print(f"[Anchor] {len(anchors):,} ✓")

    print(f"[Aggregate] building {len(qa_keys)} binary columns ...")
    q_to_skip_persons = skipped.groupby('question_concept_id')['person_id'].apply(set).to_dict()

    person_ids = anchors['person_id'].values

    new_cols = {}
    for i, key in enumerate(qa_keys):
        chose = key_to_persons.get(key, set())
        qid = key_to_qid[key]
        skipped_p = q_to_skip_persons.get(qid, set())

        col = np.zeros(len(person_ids), dtype='float32')
        if skipped_p:
            skip_mask = np.array([pid in skipped_p for pid in person_ids])
            col[skip_mask] = np.nan
        if chose:
            chose_mask = np.array([pid in chose for pid in person_ids])
            col[chose_mask] = 1.0
        new_cols[key] = col

        if (i+1) % 100 == 0:
            print(f"  {i+1}/{len(qa_keys)}")

    result = pd.concat([anchors[['person_id']].reset_index(drop=True),
                        pd.DataFrame(new_cols)], axis=1)

    total = answered.groupby('person_id').size().rename('surv_total_count').reset_index()
    unique_q = (answered.groupby('person_id')['question_concept_id']
                .nunique().rename('surv_unique_question_count').reset_index())
    skip_count = skipped.groupby('person_id').size().rename('surv_skip_count').reset_index()

    for df in [total, unique_q, skip_count]:
        result = result.merge(df, on='person_id', how='left')
    for c in ['surv_total_count', 'surv_unique_question_count', 'surv_skip_count']:
        result[c] = result[c].fillna(0).astype('int32')

    print(f"[Result] shape: {result.shape}")

    check = result.merge(anchors, on='person_id', how='left')
    cov = (check['surv_total_count'] > 0).sum()
    pm = check[check['IsPositive']==1]['surv_total_count'].mean()
    nmn = check[check['IsPositive']==0]['surv_total_count'].mean()
    print(f"\n[Sanity]")
    print(f"  surv_total_count: pos={pm:.2f} | neg={nmn:.2f} | ratio={pm/max(nmn,1e-6):.2f}x")
    print(f"  coverage: {cov:,}/{N_TOTAL} ({cov/N_TOTAL*100:.1f}%)")
    sp = check[check['IsPositive']==1]['surv_skip_count'].mean()
    sn = check[check['IsPositive']==0]['surv_skip_count'].mean()
    print(f"  surv_skip_count: pos={sp:.2f} | neg={sn:.2f}")

    if qa_keys:
        sample = qa_keys[0]
        n1 = (result[sample] == 1).sum()
        n0 = (result[sample] == 0).sum()
        nn = result[sample].isna().sum()
        print(f"  sample binary column {sample}: 1={n1:,} | 0={n0:,} | NaN={nn:,}")

    name_map = (answered_valid.groupby(['qa_key','question_concept_id','question',
                                    'answer_concept_id','answer'])['person_id']
                .nunique().reset_index(name='n_persons')
                .sort_values('n_persons', ascending=False))
    name_map = name_map.merge(q_response[['question_concept_id','response_rate']],
                               on='question_concept_id', how='left')

    bad_q = name_map.groupby('qa_key')['question_concept_id'].nunique()
    bad_a = name_map.groupby('qa_key')['answer_concept_id'].nunique()
    assert bad_q.max() == 1 and bad_a.max() == 1, "qa_key still has collision, check ids for duplicates/dirty values"
    print(f"[Check] qa_key ↔ (question_concept_id, answer_concept_id) 1:1 ✓")

    result.to_parquet(f'{FEATURE_DIR}/survey_features_{timeframe}.parquet', index=False)
    name_map.to_csv(f'{FEATURE_DIR}/survey_name_map_{timeframe}.csv', index=False)
    print(f"survey_features_{timeframe}.parquet saved")

In [ ]:
for tf in [6]:
    build_survey_features(tf)

In [ ]:
timeframe = 24
CLEAN_DIR = f'{BASE}/amia/clean_data'

s_pos = pd.read_csv(f'{CLEAN_DIR}/clean_survey_{timeframe}.csv', low_memory=False)
s_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_survey_{timeframe}.csv', low_memory=False)

all_cols = set(s_pos.columns) | set(s_neg.columns)
for col in all_cols - set(s_pos.columns): s_pos[col] = pd.NA
for col in all_cols - set(s_neg.columns): s_neg[col] = pd.NA
table = pd.concat([s_pos[sorted(all_cols)], s_neg[sorted(all_cols)]], ignore_index=True)

table['survey_datetime'] = pd.to_datetime(table['survey_datetime'], errors='coerce').dt.normalize()
table = table.dropna(subset=['survey_datetime']).copy()

is_skip = table['answer'].astype(str).str.strip().eq('PMI: Skip')
answered = table[~is_skip].copy()

ans_bad = pd.to_numeric(answered['answer_concept_id'], errors='coerce').isna()
print(f"24m total answered: {len(answered):,}")
print(f"missing answer_concept_id: {ans_bad.sum():,} ({ans_bad.mean()*100:.2f}%)")
print()
print("questions with most missing rows:")
print(answered[ans_bad]['question'].value_counts().head(15))

n_participants = answered['person_id'].nunique()
q_response_full = (answered.groupby(['question_concept_id','question'])['person_id']
                    .nunique().reset_index(name='n_answered'))
q_response_full['response_rate'] = q_response_full['n_answered'] / n_participants
missing_qids = answered[ans_bad]['question_concept_id'].unique()
print()
print("response rate of the questions that contain missing rows:")
print(q_response_full[q_response_full['question_concept_id'].isin(missing_qids)]
      .sort_values('response_rate')[['question','response_rate']])

In [ ]:
missing_summary = (
    answered.assign(ans_bad=pd.to_numeric(answered['answer_concept_id'], errors='coerce').isna())
    .groupby('question')
    .agg(total_rows=('ans_bad', 'size'),
         missing_rows=('ans_bad', 'sum'))
)
missing_summary = missing_summary[missing_summary['missing_rows'] > 0]
missing_summary['missing_pct'] = (missing_summary['missing_rows'] / missing_summary['total_rows'] * 100).round(1)
missing_summary = missing_summary.sort_values('missing_rows', ascending=False)
print(missing_summary.to_string())

for q in missing_summary.index[:8]:
    sample_vals = answered[answered['question']==q]['answer'].dropna().unique()[:8]
    print(f"\n{q}")
    print(f"  answer example: {list(sample_vals)}")

In [ ]:
bad = answered_before_dropna_here 
bad[bad['answer_concept_id'].isna()]['question'].value_counts().head(10)

In [ ]:
for tf in [12,24]:
    build_survey_features(tf)

In [ ]:
dup_answers = answered.groupby(['person_id','q_clean'])['a_clean'].nunique()
print((dup_answers > 1).sum(), "/", dup_answers.index.get_level_values(0).nunique())

ans_q = answered.groupby('person_id')['q_clean'].apply(set)
skip_q = skipped.groupby('person_id')['q_clean'].apply(set)
overlap = sum(len(ans_q.get(p,set()) & skip_q.get(p,set())) > 0 for p in set(ans_q.index)&set(skip_q.index))
print(overlap)

In [ ]:
demographic

In [ ]:
import pandas as pd

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR  = f'{BASE}/amia/clean_data'
MATRIX_DIR = f'{BASE}/amia/matrix'
N_TOTAL = 267747

demo = pd.read_csv(f'{CLEAN_DIR}/demographic.csv')
assert len(demo) == N_TOTAL
print(f"demo: +{demo.shape[1]-1} features\n")

for tf in [6, 12, 24]:
    ehr = pd.read_parquet(f'{MATRIX_DIR}/matrix_ehr_{tf}.parquet')
    m = ehr.merge(demo, on='person_id', how='left')
    assert len(m) == N_TOTAL, f"{tf}m row error: {len(m)}"
    m.to_parquet(f'{MATRIX_DIR}/matrix_ehrdemo_{tf}.parquet', index=False)
    print(f"[{tf}m] {ehr.shape[1]-2} → {m.shape[1]-2} features | {m.shape} ✓")

print("\nmatrix_ehrdemo_{6,12,24}.parquet saved")

In [ ]:
Survey updated

In [ ]:
import pandas as pd, re

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR   = f'{BASE}/amia/clean_data'
FEATURE_DIR = f'{BASE}/amia/feature'
N_TOTAL = 267747

def clean_col(text, max_len=30):
    if pd.isna(text) or not str(text).strip():
        return 'unknown'
    s = re.sub(r'_+','_', re.sub(r'[^a-z0-9]+','_', str(text).lower())).strip('_')
    return (s[:max_len].rstrip('_') if len(s) > max_len else s) or 'unknown'

DUP_QUESTIONS = [
    'Gender: Gender Identity',
    'Biological Sex At Birth: Sex At Birth',
    'Black: Black Specific', 'Hispanic: Hispanic Specific', 'AIAN: AIAN Specific',
    'MENA: MENA Specific', 'NHPI: NHPI Specific', 'Asian: Asian Specific',
    'White: White Specific',
]

for tf in [6, 12, 24]:
    print(f"\n{'='*60}\nTIMEFRAME = {tf}\n{'='*60}")

    prefixes = tuple(f'surv_{clean_col(q)}__' for q in DUP_QUESTIONS)

    surv = pd.read_parquet(f'{FEATURE_DIR}/survey_features_{tf}.parquet')
    GLOBAL = {'person_id','surv_total_count','surv_unique_question_count','surv_skip_count'}
    dropped = [c for c in surv.columns if c not in GLOBAL and c.startswith(prefixes)]
    out = surv[[c for c in surv.columns if c not in dropped]]

    print(f"keep income/home/employment examples:")
    kept_basics = [c for c in out.columns if any(k in c for k in
                   ['income','home_own','employment','education','disability','insurance'])]
    print(f"  {kept_basics[:8]}")

    assert len(out) == N_TOTAL
    out.to_csv(f'{CLEAN_DIR}/survey_nobasic_{tf}.csv', index=False)
    print(f" saved: survey_nobasic_{tf}.csv  {out.shape}")

In [ ]:
import pandas as pd

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR  = f'{BASE}/amia/clean_data'
MATRIX_DIR = f'{BASE}/amia/matrix'
N_TOTAL = 267747

for tf in [6, 12, 24]:
    ehrdemo = pd.read_parquet(f'{MATRIX_DIR}/matrix_ehrdemo_{tf}.parquet')
    surv_nb = pd.read_csv(f'{CLEAN_DIR}/survey_nobasic_{tf}.csv')
    assert len(surv_nb) == N_TOTAL, f"survey_nobasic_{tf} row error"

    m = ehrdemo.merge(surv_nb, on='person_id', how='left')
    assert len(m) == N_TOTAL

    m.to_parquet(f'{MATRIX_DIR}/matrix_ehrdemo_survnobasic_{tf}.parquet', index=False)
    print(f"[{tf}m] ehrdemo {ehrdemo.shape[1]-2} + survnobasic {surv_nb.shape[1]-1} "
          f"→ {m.shape[1]-2} features | {m.shape} ✓")

print("\n matrix_ehrdemo_survnobasic_{6,12,24}.parquet saved")